# Customer Churn Prediction — Live Portfolio Demo

**A.Masmi**

Realistic telecom customer churn machine-learning project.

This notebook loads a public Telco Customer Churn dataset, cleans and visualizes it, compares Logistic Regression, Random Forest, and Gradient Boosting, evaluates key classification metrics, selects the best model by ROC-AUC, and saves the artifacts for a Streamlit live demo.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, RocCurveDisplay
)

pd.set_option("display.max_columns", None)
print("Libraries loaded successfully.")


## 1. Load the Dataset

In [ ]:
DATA_URL = (
    "https://raw.githubusercontent.com/Giskard-AI/examples/"
    "main/datasets/WA_Fn-UseC_-Telco-Customer-Churn.csv"
)

print("Downloading Telco Customer Churn dataset...")
df = pd.read_csv(DATA_URL)
df.to_csv("telco_customer_churn_original.csv", index=False)

print("Dataset loaded successfully.")
print("Shape:", df.shape)
display(df.head())


## 2. Inspect the Dataset

In [ ]:
print("Columns:")
print(df.columns.tolist())

print("\nChurn distribution:")
display(df["Churn"].value_counts().rename_axis("Churn").to_frame("Customers"))

print("\nChurn percentage:")
display(df["Churn"].value_counts(normalize=True).mul(100).round(2).rename("Percent").to_frame())

print("\nMissing values:")
display(df.isnull().sum().sort_values(ascending=False).head(10).to_frame("Missing"))


## 3. Clean the Data

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
print("Rows with missing TotalCharges:", df["TotalCharges"].isna().sum())

df = df.dropna().copy()
df = df.drop(columns=["customerID"])
df["Churn"] = df["Churn"].map({"No": 0, "Yes": 1})

df.to_csv("telco_customer_churn_clean.csv", index=False)

print("Clean dataset shape:", df.shape)
print("Churn rate:", f"{df['Churn'].mean() * 100:.2f}%")
display(df.head())


## 4. Exploratory Data Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
counts = df["Churn"].value_counts().sort_index()
ax.bar(["Stayed", "Churned"], counts.values)
ax.set_title("Customer Churn Distribution")
ax.set_ylabel("Number of Customers")
ax.grid(axis="y", alpha=0.25)
for i, value in enumerate(counts.values):
    ax.text(i, value + 40, str(value), ha="center")
plt.show()

contract_churn = pd.crosstab(df["Contract"], df["Churn"], normalize="index") * 100
contract_churn.plot(kind="bar", figsize=(9, 5))
plt.title("Churn Percentage by Contract Type")
plt.ylabel("Percentage")
plt.xlabel("Contract")
plt.xticks(rotation=0)
plt.legend(["Stayed", "Churned"])
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 5))
plt.hist(df.loc[df["Churn"] == 0, "MonthlyCharges"], bins=25, alpha=0.6, label="Stayed")
plt.hist(df.loc[df["Churn"] == 1, "MonthlyCharges"], bins=25, alpha=0.6, label="Churned")
plt.title("Monthly Charges: Stayed vs Churned")
plt.xlabel("Monthly Charges")
plt.ylabel("Customers")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

plt.figure(figsize=(9, 5))
plt.hist(df.loc[df["Churn"] == 0, "tenure"], bins=25, alpha=0.6, label="Stayed")
plt.hist(df.loc[df["Churn"] == 1, "tenure"], bins=25, alpha=0.6, label="Churned")
plt.title("Customer Tenure: Stayed vs Churned")
plt.xlabel("Tenure (Months)")
plt.ylabel("Customers")
plt.legend()
plt.grid(alpha=0.2)
plt.show()


## 5. Prepare the ML Pipeline

In [ ]:
X = df.drop(columns=["Churn"])
y = df["Churn"]

categorical_features = X.select_dtypes(include=["object"]).columns.tolist()
numeric_features = X.select_dtypes(exclude=["object"]).columns.tolist()

print("Numeric features:", numeric_features)
print("\nCategorical features:", categorical_features)

preprocessor = ColumnTransformer([
    ("numeric", StandardScaler(), numeric_features),
    ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print("\nTraining records:", len(X_train))
print("Testing records:", len(X_test))


## 6. Train and Compare Three Models

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(
        n_estimators=350, max_depth=10, min_samples_leaf=3,
        class_weight="balanced", random_state=42, n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=150, learning_rate=0.05, max_depth=3, random_state=42
    )
}

results = []
trained_models = {}

for name, model in models.items():
    print("Training:", name)
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    pipeline.fit(X_train, y_train)

    predictions = pipeline.predict(X_test)
    probabilities = pipeline.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions),
        "F1": f1_score(y_test, predictions),
        "ROC_AUC": roc_auc_score(y_test, probabilities)
    })
    trained_models[name] = pipeline

results_df = pd.DataFrame(results).sort_values("ROC_AUC", ascending=False).reset_index(drop=True)
results_df.to_csv("model_comparison.csv", index=False)

display(results_df.style.format({
    "Accuracy": "{:.4f}", "Precision": "{:.4f}", "Recall": "{:.4f}",
    "F1": "{:.4f}", "ROC_AUC": "{:.4f}"
}))


## 7. Visual Model Comparison

In [ ]:
plot_df = results_df.set_index("Model")[["Accuracy", "Precision", "Recall", "F1", "ROC_AUC"]]
plot_df.plot(kind="bar", figsize=(12, 6))
plt.title("Customer Churn - Model Performance Comparison")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.legend(loc="lower right")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 7))
for name, pipeline in trained_models.items():
    RocCurveDisplay.from_estimator(pipeline, X_test, y_test, name=name, ax=ax)
plt.title("ROC Curves - Churn Prediction Models")
plt.grid(alpha=0.25)
plt.show()


## 8. Best Model and Confusion Matrix

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]
best_auc = float(results_df.iloc[0]["ROC_AUC"])

print("BEST MODEL:", best_model_name)
print("ROC-AUC:", round(best_auc, 4))

best_predictions = best_model.predict(X_test)
cm = confusion_matrix(y_test, best_predictions)
print("\nConfusion Matrix:")
print(cm)

fig, ax = plt.subplots(figsize=(6, 5))
ax.imshow(cm)
ax.set_title(f"Confusion Matrix - {best_model_name}")
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_xticks([0, 1], labels=["Stay", "Churn"])
ax.set_yticks([0, 1], labels=["Stay", "Churn"])
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center")
plt.show()


## 9. Save Artifacts for Streamlit

In [ ]:
joblib.dump(best_model, "churn_best_model.joblib")

metadata = {
    "best_model_name": best_model_name,
    "best_roc_auc": best_auc,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "results": results_df.to_dict(orient="records")
}
joblib.dump(metadata, "churn_model_metadata.joblib")

sample_customers = X_test.copy()
sample_customers["Actual_Churn"] = y_test.values
sample_customers["Predicted_Churn_Probability"] = best_model.predict_proba(X_test)[:, 1]
sample_customers["Predicted_Churn"] = best_model.predict(X_test)
sample_customers.head(100).to_csv("sample_customers.csv", index=False)

print("=" * 60)
print("CUSTOMER CHURN PROJECT COMPLETE")
print("=" * 60)
print("Dataset records:", len(df))
print("Features:", X.shape[1])
print("Models compared:", len(models))
print("Best model:", best_model_name)
print("Best ROC-AUC:", round(best_auc, 4))
print("\nCreated files:")
for filename in [
    "telco_customer_churn_original.csv",
    "telco_customer_churn_clean.csv",
    "model_comparison.csv",
    "churn_best_model.joblib",
    "churn_model_metadata.joblib",
    "sample_customers.csv",
]:
    print(" ✓", filename)
print("\nREADY FOR STREAMLIT DEMO")
